In [ ]:
import os
import h5py
import dxchange
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
import matplotlib as mpl
from concurrent.futures import ThreadPoolExecutor

os.makedirs('figs', exist_ok=True)

## Parameters

In [ ]:
nx, ny, nz = 2048, 2048, 2048
N_THREADS  = 16

# slice selections (kept from original plots2.ipynb)
z_slice    = 140   # z-slice at nz//2 - z_slice
y_slice_off = 32  # y-slice at ny//2 + y_slice_off

prec_path = '/data2/vnikitin/atomium_rec/20240924/AtomiumS2/rec_peter65/'
ckpt_path = '/data2/vnikitin/alcf/atomium/checkpoint_1344.h5'

## Style and utilities

In [ ]:
mpl.rcParams.update({
    "font.family":      "serif",
    "font.serif":       ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size":        22,
    "xtick.labelsize":  16,
    "ytick.labelsize":  16,
})

def mshow(a, **kwargs):
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    im   = ax.imshow(a, cmap="gray", **kwargs)
    cbar = fig.colorbar(im, fraction=0.046, pad=0.05)
    cbar.ax.tick_params(labelsize=20)
    ax.add_artist(ScaleBar(7e-3, "um", length_fraction=0.25,
                           font_properties={"family": "serif", "size": 20},
                           location="lower right"))

def mshow1(a, **kwargs):
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(a, cmap="gray", **kwargs)
    ax.axis('off')

def find_min_max(data):
    h, e = np.histogram(data[:], 1000)
    st, end = np.where(h > np.max(h) * 0.03)[0][[0, -1]]
    return e[st], e[end + 1]

def norm(a, mn, mx):
    return np.clip((a - mn) / (mx - mn), 0, 1)

## Load prec from tiff stack

In [ ]:
print("Loading prec ...")
prec = dxchange.read_tiff_stack(f'{prec_path}/r_00000.tiff', ind=np.arange(0, 2048))
prec = prec[prec.shape[0]//2 - nz//2 : prec.shape[0]//2 + nz//2,
            prec.shape[1]//2 - ny//2 : prec.shape[1]//2 + ny//2,
            prec.shape[2]//2 - nx//2 : prec.shape[2]//2 + nx//2]
print(f"prec shape: {prec.shape}")

## Load mrec from checkpoint (threaded memmap)

In [ ]:
print("Loading mrec ...")
with h5py.File(ckpt_path, 'r') as fid:
    ds          = fid['obj_re']
    shape       = ds.shape
    dt          = ds.dtype
    byte_offset = ds.id.get_offset()
    z0 = shape[0]//2 - nz//2;  z1_ = shape[0]//2 + nz//2
    y0 = shape[1]//2 - ny//2;  y1  = shape[1]//2 + ny//2
    x0 = shape[2]//2 - nx//2;  x1  = shape[2]//2 + nx//2

z_edges = np.linspace(z0, z1_, N_THREADS + 1, dtype=int)
mrec    = np.empty((nz, y1 - y0, x1 - x0), dtype='float32')

if byte_offset is not None:
    mm = np.memmap(ckpt_path, dtype=dt, mode='r', offset=byte_offset, shape=shape)
    def _load(i):
        za, zb = z_edges[i], z_edges[i + 1]
        mrec[za - z0 : zb - z0] = mm[za:zb, y0:y1, x0:x1]
    with ThreadPoolExecutor(max_workers=N_THREADS) as ex:
        list(ex.map(_load, range(N_THREADS)))
    del mm
else:
    def _load(i):
        za, zb = z_edges[i], z_edges[i + 1]
        with h5py.File(ckpt_path, 'r') as f:
            mrec[za - z0 : zb - z0] = f['obj_re'][za:zb, y0:y1, x0:x1]
    with ThreadPoolExecutor(max_workers=N_THREADS) as ex:
        list(ex.map(_load, range(N_THREADS)))

print(f"mrec shape: {mrec.shape}")

## Normalization constants

In [ ]:
cal_z = nz//2 - z_slice
cal   = slice(ny//2 - 512, ny//2 + 512)
mmin1, mmax1 = find_min_max(prec[cal_z, cal, cal])
mmin2, mmax2 = find_min_max(mrec[cal_z, cal, cal])
print(f"prec  [{mmin1:.4f}, {mmax1:.4f}]")
print(f"mrec  [{mmin2:.4f}, {mmax2:.4f}]")

## Full z-slice

In [ ]:
aa = norm(prec[nz//2 - z_slice], mmin1, mmax1)
bb = norm(mrec[nz//2 - z_slice], mmin2, mmax2)
mshow(aa,  vmax=0.95, vmin=-0.05); plt.savefig("figs/precz.png",     dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()
mshow(bb,  vmax=0.95, vmin=-0.05); plt.savefig("figs/mrecz.png",     dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()
mshow1(aa, vmax=0.95, vmin=-0.05); plt.savefig("figs/precz_raw.png", dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()
mshow1(bb, vmax=0.95, vmin=-0.05); plt.savefig("figs/mrecz_raw.png", dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()

## Full y-slice

In [ ]:
aa = norm(prec[:, ny//2 + y_slice_off], mmin1, mmax1)
bb = norm(mrec[:, ny//2 + y_slice_off], mmin2, mmax2)
mshow(aa,  vmax=0.95, vmin=-0.05); plt.savefig("figs/precy.png",     dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()
mshow(bb,  vmax=0.95, vmin=-0.05); plt.savefig("figs/mrecy.png",     dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()
mshow1(aa, vmax=0.95, vmin=-0.05); plt.savefig("figs/precy_raw.png", dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()
mshow1(bb, vmax=0.95, vmin=-0.05); plt.savefig("figs/mrecy_raw.png", dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()

## Cropped z-slices

In [ ]:
aa = norm(prec[nz//2 - z_slice], mmin1, mmax1)
bb = norm(mrec[nz//2 - z_slice], mmin2, mmax2)

s, stx, sty = 100, -80, 0
mshow1(aa[ny//2-s+sty:ny//2+s+sty, nx//2-s+stx:nx//2+s+stx], vmax=0.95, vmin=-0.05); plt.savefig("figs/preczz.png",  dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()
mshow1(bb[ny//2-s+sty:ny//2+s+sty, nx//2-s+stx:nx//2+s+stx], vmax=0.95, vmin=-0.05); plt.savefig("figs/mreczz.png",  dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()

s, stx, sty = 100, 0, -400
mshow1(aa[ny//2-s+sty:ny//2+s+sty, nx//2-s+stx:nx//2+s+stx], vmax=0.95, vmin=-0.05); plt.savefig("figs/preczz2.png", dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()
mshow1(bb[ny//2-s+sty:ny//2+s+sty, nx//2-s+stx:nx//2+s+stx], vmax=0.95, vmin=-0.05); plt.savefig("figs/mreczz2.png", dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()

## Cropped y-slices

In [ ]:
aa = norm(prec[:, ny//2 + y_slice_off], mmin1, mmax1)
bb = norm(mrec[:, ny//2 + y_slice_off], mmin2, mmax2)

s, stx, sty = 100, -120, -200
mshow1(aa[nz//2-s+sty:nz//2+s+sty, nx//2-s+stx:nx//2+s+stx], vmax=0.95, vmin=-0.05); plt.savefig("figs/precyy.png",  dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()
mshow1(bb[nz//2-s+sty:nz//2+s+sty, nx//2-s+stx:nx//2+s+stx], vmax=0.95, vmin=-0.05); plt.savefig("figs/mrecyy.png",  dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()

s, stx, sty = 100, 730, -70
mshow1(aa[nz//2-s+sty:nz//2+s+sty, nx//2-s+stx:nx//2+s+stx], vmax=0.95, vmin=-0.05); plt.savefig("figs/precyy2.png", dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()
mshow1(bb[nz//2-s+sty:nz//2+s+sty, nx//2-s+stx:nx//2+s+stx], vmax=0.95, vmin=-0.05); plt.savefig("figs/mrecyy2.png", dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()

s, stx, sty = 100, -250, 70
mshow1(aa[nz//2-s+sty:nz//2+s+sty, nx//2-s+stx:nx//2+s+stx], vmax=0.95, vmin=-0.05); plt.savefig("figs/precyy3.png", dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()
mshow1(bb[nz//2-s+sty:nz//2+s+sty, nx//2-s+stx:nx//2+s+stx], vmax=0.95, vmin=-0.05); plt.savefig("figs/mrecyy3.png", dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()

## Zoomed patches

In [ ]:
for tag, middle in [("0", [nz//2, ny//2+64,  nx//2-60]),
                    ("1", [nz//2, 2500+60,   2100+90])]:
    ss = 128
    ap = prec[middle[0]-ss:middle[0]+ss, middle[1]-ss:middle[1]+ss, middle[2]-ss:middle[2]+ss]
    bp = mrec[middle[0]-ss:middle[0]+ss, middle[1]-ss:middle[1]+ss, middle[2]-ss:middle[2]+ss]
    aa = norm(ap, mmin1, mmax1)
    bb = norm(bp, mmin2, mmax2)

    mshow1(aa[ss],        vmax=0.8, vmin=0); plt.savefig(f"figs/preczp{tag}.png", dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()
    mshow1(bb[ss],        vmax=1,   vmin=0); plt.savefig(f"figs/mreczp{tag}.png", dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()
    mshow1(aa[:, ss],     vmax=0.8, vmin=0); plt.savefig(f"figs/precyp{tag}.png", dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()
    mshow1(bb[:, ss],     vmax=1,   vmin=0); plt.savefig(f"figs/mrecyp{tag}.png", dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()
    mshow1(aa[:, :, ss],  vmax=0.8, vmin=0); plt.savefig(f"figs/precxp{tag}.png", dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()
    mshow1(bb[:, :, ss],  vmax=1,   vmin=0); plt.savefig(f"figs/mrecxp{tag}.png", dpi=300, bbox_inches="tight", pad_inches=0.02); plt.close()

print("Done — figures saved to figs/")